In [ ]:
import pandas as pd
import requests

url = "https://api.jolpi.ca/ergast/f1/2024/3/pitstops.json"
response = requests.get(url, timeout=30)
response.raise_for_status()
data = response.json()

data

In [ ]:
# JSON normalization

races = data["MRData"]["RaceTable"]["Races"]

pit_stops = pd.json_normalize(
    races,
    record_path="PitStops",
    meta=["season", "round", "raceName"],
)

pit_stops.head(10)

,driverId,lap,stop,time,duration,season,round,raceName
0,ricciardo,5,1,15:10:25,18.265,2024,3,Australian Grand Prix
1,albon,6,1,15:11:45,17.755,2024,3,Australian Grand Prix
2,zhou,6,1,15:11:49,19.032,2024,3,Australian Grand Prix
3,hamilton,7,1,15:13:04,17.596,2024,3,Australian Grand Prix
4,kevin_magnussen,7,1,15:13:08,18.301,2024,3,Australian Grand Prix
5,russell,8,1,15:14:23,18.046,2024,3,Australian Grand Prix
6,stroll,8,1,15:14:26,17.901,2024,3,Australian Grand Prix
7,bottas,8,1,15:14:30,46.969,2024,3,Australian Grand Prix
8,leclerc,9,1,15:15:42,17.409,2024,3,Australian Grand Prix
9,piastri,9,1,15:15:43,17.573,2024,3,Australian Grand Prix


In [ ]:
# Dataset inspection

print("Dataset shape:", pit_stops.shape)
print("\nColumn names:", pit_stops.columns.tolist())
print("\nData types:")
print(pit_stops.dtypes)

Dataset shape: (30, 8)

Column names: ['driverId', 'lap', 'stop', 'time', 'duration', 'season', 'round', 'raceName']

Data types:
driverId    str
lap         str
stop        str
time        str
duration    str
season      str
round       str
raceName    str
dtype: object


In [ ]:
# Data transformation

numeric_columns = ["lap", "stop", "duration"]

for column in numeric_columns:
    pit_stops[column] = pd.to_numeric(
        pit_stops[column],
        errors="coerce",
    )

print(pit_stops[numeric_columns].dtypes)

lap           int64
stop          int64
duration    float64
dtype: object


In [ ]:
# Create a calculated column

average_duration = pit_stops["duration"].mean()

pit_stops["duration_difference"] = pit_stops["duration"] - average_duration

pit_stops.head(10).round(3)

,driverId,lap,stop,time,duration,season,round,raceName,duration_difference
0,ricciardo,5,1,15:10:25,18.265,2024,3,Australian Grand Prix,-1.826
1,albon,6,1,15:11:45,17.755,2024,3,Australian Grand Prix,-2.336
2,zhou,6,1,15:11:49,19.032,2024,3,Australian Grand Prix,-1.059
3,hamilton,7,1,15:13:04,17.596,2024,3,Australian Grand Prix,-2.495
4,kevin_magnussen,7,1,15:13:08,18.301,2024,3,Australian Grand Prix,-1.790
5,russell,8,1,15:14:23,18.046,2024,3,Australian Grand Prix,-2.045
6,stroll,8,1,15:14:26,17.901,2024,3,Australian Grand Prix,-2.190
7,bottas,8,1,15:14:30,46.969,2024,3,Australian Grand Prix,26.878
8,leclerc,9,1,15:15:42,17.409,2024,3,Australian Grand Prix,-2.682
9,piastri,9,1,15:15:43,17.573,2024,3,Australian Grand Prix,-2.518


In [ ]:
# Statistical summary

statistics = pit_stops[numeric_columns].agg(["mean", "median", "std", "min", "max"]).T

statistics.columns = [
    "Mean",
    "Median",
    "Standard Deviation",
    "Minimum",
    "Maximum",
]

statistics.round(3)

,Mean,Median,Standard Deviation,Minimum,Maximum
lap,19.267,16.000,12.202,5.000,39.000
stop,1.400,1.000,0.498,1.000,2.000
duration,20.091,18.128,6.523,17.222,46.969


In [ ]:
# Overall descriptive statistics

pit_stops.describe().round(3)

,lap,stop,duration,duration_difference
count,30.000,30.000,30.000,30.000
mean,19.267,1.400,20.091,0.000
std,12.202,0.498,6.523,6.523
min,5.000,1.000,17.222,-2.869
25%,8.250,1.000,17.746,-2.345
50%,16.000,1.000,18.128,-1.963
75%,33.750,2.000,18.472,-1.619
max,39.000,2.000,46.969,26.878


In [ ]:
# Fastest and slowest pit stops: Filtering 

valid_stops = pit_stops.dropna(subset=["duration"])

fastest_stop = valid_stops.loc[[valid_stops["duration"].idxmin()]]

slowest_stop = valid_stops.loc[[valid_stops["duration"].idxmax()]]

fastest_stop.round(3)

,driverId,lap,stop,time,duration,season,round,raceName,duration_difference
23,perez,35,2,15:52:18,17.222,2024,3,Australian Grand Prix,-2.869


In [ ]:
slowest_stop.round(3)

,driverId,lap,stop,time,duration,season,round,raceName,duration_difference
7,bottas,8,1,15:14:30,46.969,2024,3,Australian Grand Prix,26.878


In [ ]:
filtered_stops_1 = pit_stops[
    (pit_stops["lap"] > 10) &
    (pit_stops["duration"] < 20)
]

filtered_stops_2 = pit_stops[
    (pit_stops["duration"] > 30) |
    (
        (pit_stops["lap"] > 30) &
        (pit_stops["duration"] < 18)
    )
]

filtered_stops_3 = pit_stops[
    (pit_stops["driverId"].isin(["albon", "ricciardo"])) &
    (pit_stops["duration"] < 18) &
    (pit_stops["lap"] < 35)
]

filtered_stops_4 = pit_stops[
    (pit_stops["lap"] >= 10) &
    (pit_stops["duration"] <= 20) &
    (pit_stops["driverId"] != "hamilton")
]

filtered_stops_1

,driverId,lap,stop,time,duration,season,round,raceName,duration_difference
12,norris,14,1,15:22:37,17.776,2024,3,Australian Grand Prix,-2.315233
13,perez,14,1,15:22:45,18.112,2024,3,Australian Grand Prix,-1.979233
14,sainz,16,1,15:25:14,17.599,2024,3,Australian Grand Prix,-2.492233
16,alonso,17,1,15:27:06,18.160,2024,3,Australian Grand Prix,-1.931233
17,hulkenberg,17,1,15:27:28,18.320,2024,3,Australian Grand Prix,-1.771233
...,...,...,...,...,...,...,...,...,...
24,hulkenberg,35,2,15:52:40,18.145,2024,3,Australian Grand Prix,-1.946233
26,tsunoda,36,2,15:53:59,17.535,2024,3,Australian Grand Prix,-2.556233
27,bottas,36,2,15:54:30,18.938,2024,3,Australian Grand Prix,-1.153233
28,stroll,37,2,15:55:19,17.977,2024,3,Australian Grand Prix,-2.114233


In [ ]:
filtered_stops_2

,driverId,lap,stop,time,duration,season,round,raceName,duration_difference
7,bottas,8,1,15:14:30,46.969,2024,3,Australian Grand Prix,26.877767
15,ocon,16,2,15:25:58,30.626,2024,3,Australian Grand Prix,10.534767
22,leclerc,34,2,15:50:35,17.308,2024,3,Australian Grand Prix,-2.783233
23,perez,35,2,15:52:18,17.222,2024,3,Australian Grand Prix,-2.869233
25,zhou,35,2,15:52:50,36.866,2024,3,Australian Grand Prix,16.774767
26,tsunoda,36,2,15:53:59,17.535,2024,3,Australian Grand Prix,-2.556233
28,stroll,37,2,15:55:19,17.977,2024,3,Australian Grand Prix,-2.114233


In [ ]:
filtered_stops_3

,driverId,lap,stop,time,duration,season,round,raceName,duration_difference
1,albon,6,1,15:11:45,17.755,2024,3,Australian Grand Prix,-2.336233
19,albon,27,2,15:41:35,17.745,2024,3,Australian Grand Prix,-2.346233
20,ricciardo,29,2,15:44:29,17.751,2024,3,Australian Grand Prix,-2.340233


In [ ]:
filtered_stops_4

,driverId,lap,stop,time,duration,season,round,raceName,duration_difference
12,norris,14,1,15:22:37,17.776,2024,3,Australian Grand Prix,-2.315233
13,perez,14,1,15:22:45,18.112,2024,3,Australian Grand Prix,-1.979233
14,sainz,16,1,15:25:14,17.599,2024,3,Australian Grand Prix,-2.492233
16,alonso,17,1,15:27:06,18.160,2024,3,Australian Grand Prix,-1.931233
17,hulkenberg,17,1,15:27:28,18.320,2024,3,Australian Grand Prix,-1.771233
...,...,...,...,...,...,...,...,...,...
24,hulkenberg,35,2,15:52:40,18.145,2024,3,Australian Grand Prix,-1.946233
26,tsunoda,36,2,15:53:59,17.535,2024,3,Australian Grand Prix,-2.556233
27,bottas,36,2,15:54:30,18.938,2024,3,Australian Grand Prix,-1.153233
28,stroll,37,2,15:55:19,17.977,2024,3,Australian Grand Prix,-2.114233
